<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.1-architecture-patterns/notebooks/exercises-10_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.1: Architecture Patterns for AI Apps

*Module 10 · AI System Architecture & Production Deployment*

> Most AI tutorials end at model.generate() . Production AI has: API gateways, model routers, provider fallbacks, queue-based processing, and streaming fan-out. This lesson teaches 5 architecture patterns that every production GenAI system uses — each diagrammed, explained, and coded as a working FastAPI service.

`Gateway` · `Router` · `Fallback` · `Queue` · `Streaming` · `75 min`

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-10.1.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Pattern 1: API Gateway — Single Entry Point — `01_gateway.py`
2. Step 2 — Pattern 2: Model Router — Pick the Right Model per Request — `02_router.py`
3. Step 3 — Pattern 3: Provider Fallback — Never Go Down — `03_fallback.py`
4. Step 4 — Pattern 4: Queue-Based Processing — Decouple and Scale — `04_queue_pattern.py`
5. Step 5 — Pattern 5: Streaming Fan-Out — Real-Time Token Delivery — `05_streaming.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai openai fastapi


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Pattern 1: API Gateway — Single Entry Point

One URL for all AI features. Auth, rate limiting, logging, routing — all in one layer.

**`01_gateway.py`** · _Block 1 of 5_

API GATEWAY — Single entry point for all AI services


In [ ]:
# API GATEWAY — Single entry point for all AI services
from fastapi import FastAPI, Request, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import time, hashlib
from collections import defaultdict

app = FastAPI(title="Netsetos AI Gateway")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"])

# ── Rate limiter ──
rate_limits = defaultdict(list)
def check_rate(api_key, limit=60, window=60):
    now = time.time()
    rate_limits[api_key] = [t for t in rate_limits[api_key] if now-t < window]
    if len(rate_limits[api_key]) >= limit: return False
    rate_limits[api_key].append(now); return True

# ── Auth middleware ──
@app.middleware("http")
async def auth_and_log(request: Request, call_next):
    start = time.time()
    api_key = request.headers.get("X-API-Key", "anonymous")

    if not check_rate(api_key):
        raise HTTPException(429, "Rate limit exceeded")

    response = await call_next(request)
    latency = (time.time()-start)*1000
    print(f"[{api_key[:8]}] {request.method} {request.url.path} {response.status_code} {latency:.0f}ms")
    response.headers["X-Latency-Ms"] = str(int(latency))
    return response

# ── Routes to AI services ──
@app.post("/v1/chat")
async def chat(request: Request):
    body = await request.json()
    # Forward to Gemini / LLM service
    return {"response": f"Chat: {body.get('message','')[:50]}", "model": "gemini-2.0-flash"}

@app.post("/v1/embed")
async def embed(request: Request):
    return {"embedding": [0.1]*768, "model": "text-embedding-004"}

@app.get("/")
async def health(): return {"status":"healthy", "service":"netsetos-gateway"}

print("API Gateway Pattern:\n")
print("  Single URL for all AI features")
print("  Auth + rate limiting + logging in middleware")
print("  Routes: /v1/chat, /v1/embed, /v1/image, /v1/agent")
print("  Deploy: one Cloud Run service = your AI API")


### Step 2 · Pattern 2: Model Router — Pick the Right Model per Request

Simple queries get the fast model. Complex ones get the powerful model. Save 60% on costs.

**`02_router.py`** · _Block 2 of 5_

MODEL ROUTER — Route to the right model per request


In [ ]:
# MODEL ROUTER — Route to the right model per request
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class ModelRouter:
    """Route simple queries to Flash, complex to Pro. Save 60% on costs."""
    MODELS = {
        "fast": {"name":"gemini-2.0-flash", "cost_per_1k":0.00015},
        "smart": {"name":"gemini-2.5-pro", "cost_per_1k":0.00125},
    }

    def __init__(self):
        self.classifier = genai.GenerativeModel("gemini-2.0-flash")

    def route(self, query):
        """Classify query complexity, pick model."""
        resp = self.classifier.generate_content(
            f"Classify complexity: SIMPLE or COMPLEX.\n"
            f"SIMPLE: greetings, factual lookup, short answers.\n"
            f"COMPLEX: reasoning, analysis, multi-step, code generation.\n"
            f"Query: {query}\nReturn ONLY: SIMPLE or COMPLEX")
        complexity = resp.text.strip().upper()
        model_key = "fast" if "SIMPLE" in complexity else "smart"
        return self.MODELS[model_key]

    def generate(self, query):
        model_cfg = self.route(query)
        model = genai.GenerativeModel(model_cfg["name"])
        resp = model.generate_content(query)
        return {"response":resp.text.strip(), "model":model_cfg["name"], "cost":model_cfg["cost_per_1k"]}

router = ModelRouter()
print("Model Router:\n")
for q in ["Hi!", "What is the refund policy?", "Compare GenAI course ROI vs a masters degree"]:
    r = router.route(q)
    print(f"  '{q[:45]}' → {r['name']} (${r['cost_per_1k']}/1k)")
print(f"\n  Flash: $0.15/M tokens. Pro: $1.25/M. Router saves ~60% on mixed traffic.")


### Step 3 · Pattern 3: Provider Fallback — Never Go Down

If Gemini fails, fall back to OpenAI. If OpenAI fails, fall back to Claude. Zero downtime.

**`03_fallback.py`** · _Block 3 of 5_

PROVIDER FALLBACK — Automatic failover across LLM providers


In [ ]:
# PROVIDER FALLBACK — Automatic failover across LLM providers
import time

class LLMFallback:
    """Try providers in order. First success wins."""
    def __init__(self, providers):
        self.providers = providers  # [{name, call_fn, priority}]
        self.stats = {p["name"]: {"success":0,"fail":0} for p in providers}

    def call(self, prompt, timeout=10):
        for p in sorted(self.providers, key=lambda x: x["priority"]):
            try:
                start = time.time()
                result = p["call_fn"](prompt)
                latency = (time.time()-start)*1000
                self.stats[p["name"]]["success"] += 1
                return {"text":result, "provider":p["name"], "ms":latency}
            except Exception as e:
                self.stats[p["name"]]["fail"] += 1
                print(f"  [{p['name']}] failed: {e}. Trying next...")
        return {"text":"All providers failed", "provider":"none"}

# Simulated providers
def gemini_call(p): return f"Gemini: {p[:30]}"
def openai_call(p): return f"OpenAI: {p[:30]}"
def claude_call(p): return f"Claude: {p[:30]}"

llm = LLMFallback([
    {"name":"gemini", "call_fn":gemini_call, "priority":1},
    {"name":"openai", "call_fn":openai_call, "priority":2},
    {"name":"claude", "call_fn":claude_call, "priority":3},
])

print("Provider Fallback:\n")
r = llm.call("What is the refund policy?")
print(f"  Result: {r['text']} (via {r['provider']}, {r['ms']:.0f}ms)")
print(f"  Stats: {llm.stats}")
print(f"\n  If Gemini is down: auto-routes to OpenAI")
print(f"  If both down: routes to Claude")
print(f"  Zero downtime. Users never see an error.")


### Step 4 · Pattern 4: Queue-Based Processing — Decouple and Scale

Accept requests instantly. Process asynchronously. Scale workers independently.

**`04_queue_pattern.py`** · _Block 4 of 5_

QUEUE-BASED PROCESSING — Decouple intake from processing


In [ ]:
# QUEUE-BASED PROCESSING — Decouple intake from processing
from fastapi import FastAPI, BackgroundTasks
import uuid, json, time, asyncio
from collections import deque

app = FastAPI()
jobs = {}  # In production: Redis or Firestore
queue = deque()

@app.post("/generate")
async def submit(request: dict, bg: BackgroundTasks):
    """Accept instantly, process in background."""
    job_id = str(uuid.uuid4())[:8]
    jobs[job_id] = {"status":"queued", "created":time.time()}
    bg.add_task(process_job, job_id, request.get("prompt",""))
    return {"job_id":job_id, "status":"queued"}

@app.get("/status/{job_id}")
async def status(job_id: str):
    return jobs.get(job_id, {"error":"Not found"})

async def process_job(job_id, prompt):
    jobs[job_id]["status"] = "processing"
    await asyncio.sleep(2)  # Simulate LLM call
    jobs[job_id].update({"status":"completed", "result":f"Generated: {prompt[:50]}"})

print("Queue Pattern:\n")
print("  POST /generate → job_id (instant, <50ms)")
print("  GET /status/{id} → queued|processing|completed")
print("  Benefits: handles traffic spikes, workers scale independently")
print("  Production: replace deque with Pub/Sub or Cloud Tasks")


### Step 5 · Pattern 5: Streaming Fan-Out — Real-Time Token Delivery

Stream LLM tokens to multiple clients simultaneously via SSE.

**`05_streaming.py`** · _Block 5 of 5_

STREAMING FAN-OUT — SSE token delivery


In [ ]:
# STREAMING FAN-OUT — SSE token delivery
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import asyncio, json

app = FastAPI()

async def generate_stream(prompt):
    """Simulate streaming LLM response token by token."""
    tokens = f"Here is the answer to: {prompt}".split()
    for token in tokens:
        yield f"data: {json.dumps({'token': token})}\n\n"
        await asyncio.sleep(0.1)  # Simulate token generation
    yield f"data: {json.dumps({'done': True})}\n\n"

@app.get("/stream")
async def stream(prompt: str = "Hello"):
    return StreamingResponse(generate_stream(prompt), media_type="text/event-stream")

print("Streaming Fan-Out:\n")
print("  GET /stream?prompt=Hello → SSE (Server-Sent Events)")
print("  Each token: data: {\"token\": \"word\"}")
print("  Final:      data: {\"done\": true}")
print("  Client: EventSource('/stream?prompt=...') in JavaScript")
print("  Time-to-first-token: ~100ms vs ~2s for full response")


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
